# Transformation explorer

Plots a TIMIT WAV file's raw waveform next to its spectrogram, featurized like `src/debug_items.py` does
(`compute_mel_spectrogram` / `compute_linear_spectrogram` from `src/data/features.py`, then `apply_drc`).
The widgets start at the `TransformationConfig` defaults; changing any of them (or the WAV file) re-does the transformation.

The second row shows what a model is fed:

- **Normalized utterance.** The whole utterance after `featurize_waveform`, under the selected `normalization`:
  - `frame`: per-frame z-score. This also removes each frame's level, i.e. the energy contour.
  - `segment`: each band's mean over the drawn segment is subtracted, which removes gain and channel but keeps the energy contour.
  - `none`: no normalization.

  The frames the drawn segment uses are marked.
- **Drawn segment.** One segment of `segment_duration`, drawn with the selected strategy from `DRAW_STRATEGIES` (`src/data/segments.py`), shown exactly as the model receives it, so `segment` normalization is applied here:
  - OS, SS, SU, MIX or RANDBS.
  - BS{b}: an OS crop cut into blocks of b frames, the block order permuted, the order inside each block kept.
  - GO{b} (the SST bracket's control C_b): blocks of b frames with a gap of `gap ratio` × b frames (`EvaluationConfig.sst_gap_ratio`) between them, kept in their original order. The marks on the normalized utterance show the gaps between the blocks.
  - GS{b} (S_b): GO{b}'s blocks in random order. With the same seed, GO{b} and GS{b} take the same blocks, so switching between them shows only the reordering.

  The seed sets the random position, as for a training draw. Evaluation (`_eval_windows`) instead places its windows at evenly spaced positions.
- **Shuffle pattern.** Which original frame each segment position comes from. Each block of BS, GO or GS appears as a rising run of points. GO's runs rise from left to right with jumps between them (the gaps); BS's and GS's runs are in a different order.

In [7]:
import html
import sys
from dataclasses import replace
from functools import partial
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import soundfile as sf
from IPython.display import Audio, display
from plotly.subplots import make_subplots
import plotly.io as pio; pio.renderers.default = "notebook"

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import DataConfig, EvaluationConfig, TransformationConfig
from src.data.dataset import featurize_waveform
from src.data.features import apply_drc, compute_linear_spectrogram, compute_mel_spectrogram
from src.data.segments import DRAW_STRATEGIES, gapped_span

In [8]:
timit_root = Path("~/.cache/datasets/timit").expanduser()
# Labelled relative to the TIMIT/ folder, e.g. "TRAIN/DR1/FCJF0/SA1.WAV".
wav_paths = [(str(p.relative_to(p.parents[3])), p) for p in sorted(timit_root.rglob("*")) if p.suffix.lower() == ".wav"]
len(wav_paths)

6300

In [9]:
defaults = TransformationConfig()
wav_path = widgets.Dropdown(options=wav_paths, description="WAV file", layout=widgets.Layout(width="500px"))
frame_length_s = widgets.FloatText(value=defaults.frame_length_s, step=0.001, description="frame_length_s")
frame_step_s = widgets.FloatText(value=defaults.frame_step_s, step=0.001, description="frame_step_s")
nfft = widgets.IntText(value=defaults.nfft, description="nfft")
n_mels = widgets.IntText(value=defaults.n_mels, description="n_mels")
fmin = widgets.BoundedFloatText(value=defaults.fmin, min=0, max=defaults.sample_rate, description="fmin")
fmax = widgets.BoundedFloatText(value=defaults.fmax, min=defaults.fmin, max=defaults.sample_rate, description="fmax")
type_ = widgets.Dropdown(options=[("mel", "mel"), ("lin", "linear")], value=defaults.type, description="type")
window = widgets.Dropdown(options=["hann", "hamming"], value=defaults.window, description="window")
normalization = widgets.Dropdown(options=["frame", "segment", "none"], value=defaults.normalization, description="normalization")
strategy = widgets.Dropdown(options=list(DRAW_STRATEGIES), value="OS", description="strategy")
segment_duration = widgets.BoundedFloatText(value=DataConfig().segment_duration, min=0.1, step=0.1, description="segment [s]")
seed = widgets.IntText(value=0, description="seed")
gap_ratio = widgets.BoundedFloatText(value=EvaluationConfig().sst_gap_ratio, min=0, step=0.25, description="gap ratio")

# Keep fmin <= fmax: each bounds the other.
widgets.dlink((fmin, "value"), (fmax, "min"))
widgets.dlink((fmax, "value"), (fmin, "max"))


# The figure (as an iframe) and the audio player go into HTML widgets: plotly's and Audio's own
# notebook outputs don't render inside an Output widget in PyCharm.
plot = widgets.HTML()
player = widgets.HTML()


def show(wav_path, frame_length_s, frame_step_s, nfft, n_mels, fmin, fmax, type_, window,
         normalization, strategy, segment_duration, seed, gap_ratio):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    assert sample_rate == defaults.sample_rate, f"{wav_path} is {sample_rate} Hz, config expects {defaults.sample_rate} Hz"
    config = replace(defaults, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft,
                     n_mels=n_mels, fmin=fmin, fmax=fmax, type=type_, window=window, normalization=normalization)
    # FFT magnitude of every frame (the linear spectrogram); the mel spectrogram is computed from it.
    stft = apply_drc(compute_linear_spectrogram(waveform, config))
    if type_ == "linear":
        features = stft
    else:
        features = apply_drc(compute_mel_spectrogram(waveform, config))

    # What the model is fed: the utterance as featurize_waveform normalizes it ("frame" only), then one
    # segment drawn with `strategy`. Drawing from an array of frame indices with an identically seeded
    # Generator makes the same random choices, which yields the segment's source frames.
    normalized = featurize_waveform(waveform, config)
    segment_length = DataConfig(segment_duration=segment_duration).segment_length(config)
    if normalized.shape[0] <= segment_length:
        plot.value = f"<b>The utterance ({normalized.shape[0]} frames) must be longer than one segment ({segment_length} frames).</b>"
        player.value = ""
        return
    draw = DRAW_STRATEGIES[strategy]
    if strategy.startswith(("GO", "GS")):
        # As trainer._eval_windows converts evaluation.sst_gap_ratio.
        gap = round(gap_ratio * int(strategy[2:]))
        span = gapped_span(segment_length, int(strategy[2:]), gap)
        if normalized.shape[0] < span:
            plot.value = f"<b>The utterance ({normalized.shape[0]} frames) is shorter than {strategy}'s gapped span ({span} frames).</b>"
            player.value = ""
            return
        draw = partial(draw, gap=gap)
    source_frames = draw(np.arange(normalized.shape[0])[:, None], segment_length, np.random.default_rng(seed))[:, 0]
    segment = draw(normalized, segment_length, np.random.default_rng(seed))
    if normalization == "segment":
        # As src/models/registry.py's SegmentMeanNormalization does to every model input. The utterance gets the
        # same band means subtracted, so the drawn frames have the same values in both panels.
        band_means = segment.mean(axis=0, keepdims=True)
        segment = segment - band_means
        normalized = normalized - band_means

    # Plotly, so hovering shows the exact values at the mouse position.
    title = f"waveform: {wav_path.relative_to(wav_path.parents[3])}"
    freq_label = "mel bin" if type_ == "mel" else "freq bin"
    frame_times = np.arange(features.shape[0]) * config.frame_step_s
    fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.1, vertical_spacing=0.16, subplot_titles=[
        title, f"FFT per frame: {stft.shape[0]} frames x {stft.shape[1]} bins",
        f"{type_} spectrogram {features.shape[0]} frames x {features.shape[1]} bins",
        f"normalized ({normalization}), frames of the drawn segment marked",
        f"{strategy} segment as the model sees it: {segment.shape[0]} frames",
        "shuffle pattern: source frame per segment position"])

    def colorbar(row, col, title):
        """A colour bar right next to subplot (row, col), as tall as it."""
        subplot = fig.get_subplot(row, col)
        (_, x1), (y0, y1) = subplot.xaxis.domain, subplot.yaxis.domain
        return dict(title_text=title, x=x1 + 0.005, xanchor="left", y=(y0 + y1) / 2, yanchor="middle", len=y1 - y0,
                    thickness=12)

    fig.add_trace(go.Scattergl(
        x=np.arange(len(waveform)) / sample_rate, y=waveform, mode="lines", line_width=0.5, showlegend=False,
        hovertemplate="time: %{x:.4f} s<br>amplitude: %{y:.4f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=stft.T, x=frame_times, y=np.arange(stft.shape[1]) * sample_rate / config.nfft,
        colorscale="Magma", colorbar=colorbar(1, 2, "DRC level"),
        hovertemplate="time: %{x:.3f} s<br>frequency: %{y:.1f} Hz<br>DRC level: %{z:.2f}<extra></extra>",
    ), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=features.T, x=frame_times, y=np.arange(features.shape[1]),
        colorscale="Magma", colorbar=colorbar(1, 3, "DRC level"),
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>DRC level: %{{z:.2f}}<extra></extra>",
    ), row=1, col=3)
    # Both normalized heatmaps share one colour scale, so equal values get equal colours. Normalized features
    # are centred on 0 (diverging colours); without normalization they're DRC levels.
    level = dict(colorscale="Magma") if normalization == "none" else dict(colorscale="RdBu_r", cmid=0)
    level_name = "DRC level" if normalization == "none" else "normalized level"
    fig.update_layout(coloraxis=dict(**level, colorbar=colorbar(2, 2, level_name)))
    fig.add_trace(go.Heatmap(
        z=normalized.T, x=frame_times, y=np.arange(normalized.shape[1]), coloraxis="coloraxis",
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Scattergl(
        x=np.unique(source_frames) * config.frame_step_s, y=np.full(len(np.unique(source_frames)), -2),
        mode="markers", marker=dict(color="lime", size=4, symbol="line-ns-open"), showlegend=False,
        hovertemplate="drawn frame at %{x:.3f} s<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Heatmap(
        z=segment.T, x=np.arange(segment.shape[0]), y=np.arange(segment.shape[1]), coloraxis="coloraxis",
        customdata=np.tile(source_frames, (segment.shape[1], 1)),
        hovertemplate=f"position: %{{x}}<br>source frame: %{{customdata}}<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=2)
    fig.add_trace(go.Scattergl(
        x=np.arange(len(source_frames)), y=source_frames, mode="markers", marker=dict(size=4), showlegend=False,
        hovertemplate="position: %{x}<br>source frame: %{y}<extra></extra>",
    ), row=2, col=3)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=2)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=3)
    fig.update_yaxes(title_text=freq_label, row=2, col=1)
    fig.update_yaxes(title_text=freq_label, row=2, col=2)
    fig.update_yaxes(title_text="source frame", row=2, col=3)
    if type_ == "mel":
        # The part of the FFT spectrum the mel filterbank spreads its filters over.
        for name, f in [("fmin", fmin), ("fmax", fmax)]:
            fig.add_hline(y=f, line=dict(color="cyan", width=1, dash="dash"), annotation_text=name,
                          annotation_font_color="cyan", row=1, col=2)
    fig.update_xaxes(title_text="time [s]", row=1)
    fig.update_xaxes(title_text="time [s]", row=2, col=1)
    fig.update_yaxes(title_text="amplitude", row=1, col=1)
    fig.update_yaxes(title_text="frequency [Hz]", row=1, col=2)
    fig.update_yaxes(title_text=freq_label, row=1, col=3)
    fig.update_layout(height=800, margin=dict(l=60, r=80, t=40, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 820px; border: 0"></iframe>'
    player.value = Audio(waveform, rate=sample_rate)._repr_html_()


output = widgets.interactive_output(show, dict(
    wav_path=wav_path, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft, n_mels=n_mels,
    fmin=fmin, fmax=fmax, type_=type_, window=window,
    normalization=normalization, strategy=strategy, segment_duration=segment_duration, seed=seed, gap_ratio=gap_ratio,
))
display(widgets.VBox([
    wav_path,
    widgets.HBox([frame_length_s, frame_step_s, nfft, n_mels]),
    widgets.HBox([fmin, fmax, type_, window]),
    widgets.HBox([normalization, strategy, segment_duration, seed, gap_ratio]),
    plot,
    player,
    output,
]))


## OC, GC, GO and GS of a VoxCeleb1-H utterance

How the SST bracket's gapped test inputs are cut from one utterance of the VoxCeleb1-H trial list (`list_test_hard2.txt`, the reported SV of the `pretrained-*-voxceleb-*` configs). Each input is built by `trainer._eval_windows` exactly as in evaluation: one window per utterance, its span of `gapped_span(segment, b, g)` frames centred in the utterance (position 0.5), with g = round(`gap ratio` × b) frames. The front-end sets the frame grid the inputs are cut on: WeSpeaker's fbank (ResNet34, 10 ms frames) or w2v-BERT 2.0's stacked fbank (20 ms frames).

- **Utterance (top).** The featurized utterance. The span lies between the black lines and GO/GS's gaps are shaded. The rugs below the spectrogram show which frames each input takes.
- **The four inputs (middle)**, as the model receives them, each with `segment` frames. Dotted lines mark joins, i.e. two frames meeting that weren't neighbours in the utterance:
  - OC{b}: the span's contiguous middle. An OS window of as many frames, at the same place and on the same trials. It has no join.
  - GC{b}: the span's first half-segment and its last, with all of GO's gap frames removed as one cut, so it has a single join.
  - GO{b}: blocks of b frames with g frames skipped between them, in their original order.
  - GS{b}: GO{b}'s blocks in a random order (`seed`), with no block left in place. These are the same blocks as GO{b}, only reordered.
- **Source frame per position.** The time in the utterance that each input position comes from.
- **Join roughness (bottom).** For each input, the distribution of the Euclidean distance between the two frames at each of its joins, as `temporal_probes.join_discontinuity` measures it, in the features the model is fed (`segment` normalization's band means cancel in the difference). The grey histogram behind each is the reference: the distances between every two consecutive frames of the span, i.e. the natural frame-to-frame change. OC has no join, so only the reference is shown. GC has a single join.
  The line below the figure compares GO's and GS's joins: their medians and a two-sided Mann-Whitney U test, with P(GS > GO), the probability that a random GS join is rougher than a random GO join (0.5: equally rough). This covers one utterance and a few joins, so the test has little power (with b = 64, one join each). `doc/sst-roughness-strata.ipynb` checks the same thing over all trial utterances, against the EER.
- **Players.** The first plays the whole utterance. The four below it play OC, GC, GO and GS: each input position contributes the frame period (10 or 20 ms) of samples starting where its source frame starts. Contiguous runs therefore sound like the original, and every join is an audible hard cut.
  The last row plays each input resynthesized from its feature frames, i.e. exactly what the model is fed: the utterance normalization is undone, the mel filterbank inverted per frame (non-negative least squares) and the phase estimated with Griffin-Lim. The 25 ms windows overlap every 10 ms, so a join blends its two frames over about 25 ms, an abrupt change without a click, as the model sees it. Losing the phase and the fine spectral detail above ~1 kHz makes all four sound somewhat buzzy and muffled alike.

An utterance shorter than the span has no such window, and its trials aren't scored for that b.

In [6]:
import torch
import torchaudio
from scipy.optimize import nnls
from scipy.signal import lfilter
from scipy.stats import mannwhitneyu

from src.data.features import _w2vbert_feature_extractor
from src.data.segments import BLOCK_SIZES
from src.training.trainer import _eval_windows

vox1_root = Path("~/.cache/datasets/voxceleb1").expanduser()
# The utterances of the VoxCeleb1-H trial list (voxceleb.eval_trial_meta_url), on which the SST runs report SV.
with open(vox1_root / "list_test_hard2.txt") as f:
    vox1h_paths = sorted({path for line in f for path in line.split()[1:]})
vox1h_by_speaker = {}
for path in vox1h_paths:
    vox1h_by_speaker.setdefault(path.split("/")[0], []).append(path)

# The pretrained models' first frame grids, what the draws cut (configs/pretrained-*-voxceleb-2s.yaml).
sst_front_ends = {"wespeaker_fbank (ResNet34, 10 ms frames)": "wespeaker_fbank", "w2vbert (w2v-BERT 2.0, 20 ms frames)": "w2vbert"}
sst_strategies = ("OC", "GC", "GO", "GS")

sst_speaker = widgets.Dropdown(options=sorted(vox1h_by_speaker), description="speaker")
sst_utterance = widgets.Dropdown(options=vox1h_by_speaker[sst_speaker.value], description="utterance",
                                 layout=widgets.Layout(width="400px"))
sst_speaker.observe(lambda change: setattr(sst_utterance, "options", vox1h_by_speaker[change.new]), names="value")
sst_front_end = widgets.Dropdown(options=sst_front_ends, description="front-end", layout=widgets.Layout(width="400px"))
sst_block_size = widgets.Dropdown(options=BLOCK_SIZES, value=8, description="block b")
sst_gap_ratio = widgets.BoundedFloatText(value=1.25, min=0, step=0.25, description="gap ratio")  # the pretrained 2s configs'
sst_segment_duration = widgets.BoundedFloatText(value=2.0, min=0.1, step=0.1, description="segment [s]")
sst_seed = widgets.IntText(value=0, description="seed (GS)")

sst_plot = widgets.HTML()
# GO's and GS's join roughness compared, below the figure.
sst_roughness = widgets.HTML()
sst_player = widgets.HTML()
# One player per input: what each of OC/GC/GO/GS sounds like.
sst_input_players = {name: widgets.HTML() for name in sst_strategies}
# And one per input resynthesized from its feature frames: what the model is fed, joins included.
sst_resynth_players = {name: widgets.HTML() for name in sst_strategies}


def resynthesize(frames, waveform, front_end):
    """A waveform whose features are approximately `frames`, an input cut from the features of the utterance
    `waveform`. Undoes the front-end step by step: the utterance normalization (its statistics come from the
    whole utterance, so this is exact), the mel filterbank (NNLS per frame, lossy), the phase (Griffin-Lim),
    and Kaldi's 16-bit scaling and pre-emphasis."""
    if front_end == "wespeaker_fbank":
        # As compute_wespeaker_fbank, before its mean over the utterance is subtracted.
        raw = torchaudio.compliance.kaldi.fbank(torch.from_numpy(waveform).unsqueeze(0) * (1 << 15), num_mel_bins=80,
                                                dither=0.0, window_type="hamming").numpy()
        log_mel = frames + raw.mean(axis=0)
        mel_filters, _ = torchaudio.compliance.kaldi.get_mel_banks(80, 512, 16000, 20.0, 0.0, 100.0, -500.0, 1.0)
        mel_filters = torch.nn.functional.pad(mel_filters, (0, 1)).T.numpy()  # (257, 80): Kaldi drops Nyquist
        window = torch.hamming_window(400, periodic=False, dtype=torch.float64)
    else:
        # w2v-BERT: unstack the 2 fbank frames per 20 ms frame, then undo the per-bin z-score over the utterance.
        extractor = _w2vbert_feature_extractor()
        raw = extractor._extract_fbank_features(waveform)
        log_mel = frames.reshape(-1, 80) * np.sqrt(raw.var(axis=0, ddof=1) + 1e-7) + raw.mean(axis=0)
        mel_filters = extractor.mel_filters  # (257, 80)
        window = torch.hann_window(400, periodic=False, dtype=torch.float64) ** 0.85  # Kaldi's povey window
    power = np.stack([nnls(mel_filters.T, frame)[0] for frame in np.exp(log_mel)])  # (10 ms frames, 257)
    torch.manual_seed(0)  # Griffin-Lim's random initial phase
    audio = torchaudio.functional.griffinlim(
        torch.from_numpy(np.sqrt(power).T), window, n_fft=512, hop_length=160, win_length=400, power=1.0,
        n_iter=100, momentum=0.99, length=None, rand_init=True).numpy()
    # Griffin-Lim centres frame i on sample 160 i, Kaldi's frame i starts there: shift by half a window.
    audio = np.pad(audio, (200, 0))
    return lfilter([1.0], [1.0, -0.97], audio / 2 ** 15).astype(np.float32)


def show_sst(utterance, front_end, block_size, gap_ratio, segment_duration, seed):
    waveform, sample_rate = sf.read(str(vox1_root / "wav" / utterance), dtype="float32")
    config = TransformationConfig(type=front_end, frame_length_s=0.025, frame_step_s=0.01)
    assert sample_rate == config.sample_rate, f"{utterance} is {sample_rate} Hz, config expects {config.sample_rate} Hz"
    features = featurize_waveform(waveform, config)
    period = config.frame_period_s
    segment_length = DataConfig(segment_duration=segment_duration).segment_length(config)
    gap = round(gap_ratio * block_size)  # as trainer._eval_windows converts evaluation.sst_gap_ratio
    span = gapped_span(segment_length, block_size, gap)
    sst_player.value = Audio(waveform, rate=sample_rate)._repr_html_()
    if features.shape[0] < span:
        for input_player in (*sst_input_players.values(), *sst_resynth_players.values()):
            input_player.value = ""
        sst_roughness.value = ""
        sst_plot.value = (f"<b>The utterance ({features.shape[0]} frames) is shorter than the gapped span ({span} frames): "
                          f"it has no OC/GC/GO/GS window and its trials aren't scored for b = {block_size}.</b>")
        return

    # Each input exactly as the evaluation draws it (one window in the middle of the utterance). Drawing from an
    # array of frame indices with an identically seeded Generator yields each input's source frames.
    inputs, sources = {}, {}
    for name in sst_strategies:
        strategy = f"{name}{block_size}"
        inputs[name] = _eval_windows(features, segment_length, segment_length, 1, strategy,
                                     np.random.default_rng(seed), gap_ratio)[0]
        sources[name] = _eval_windows(np.arange(features.shape[0])[:, None], segment_length, segment_length, 1,
                                      strategy, np.random.default_rng(seed), gap_ratio)[0][:, 0]

    # Each input as audio: every source frame contributes the frame period of samples starting where that frame
    # starts, so a contiguous run plays the original signal and each join is a hard cut. normalize=False keeps the
    # four players at the utterance's level, so they're comparable.
    hop = round(period * sample_rate)
    for name in sst_strategies:
        audio = waveform[(sources[name][:, None] * hop + np.arange(hop)).ravel()]
        sst_input_players[name].value = (f"<b>{name}{block_size}</b> ({len(audio) / sample_rate:.2f} s)"
                                         + Audio(audio, rate=sample_rate, normalize=False)._repr_html_())
        audio = resynthesize(inputs[name], waveform, front_end)
        sst_resynth_players[name].value = (f"<b>{name}{block_size} resynthesized</b> ({len(audio) / sample_rate:.2f} s)"
                                           + Audio(audio, rate=sample_rate, normalize=False)._repr_html_())

    num_blocks = -(-segment_length // block_size)
    start = round(0.5 * (features.shape[0] - span))  # src/data/segments.py's position 0.5
    cut = (num_blocks - 1) * gap

    # Join roughness, as temporal_probes.join_discontinuity measures it: the distance between the two frames at each
    # join (two positions whose source frames weren't neighbours). The reference is every frame-to-frame change
    # within the span.
    joins = {name: np.flatnonzero(np.diff(sources[name]) != 1) + 1 for name in sst_strategies}
    roughness = {name: np.linalg.norm(inputs[name][joins[name]] - inputs[name][joins[name] - 1], axis=1)
                 for name in sst_strategies}
    adjacent = np.linalg.norm(np.diff(features[start:start + span], axis=0), axis=1)
    go_joins, gs_joins = roughness["GO"], roughness["GS"]
    if len(go_joins) and len(gs_joins):
        test = mannwhitneyu(gs_joins, go_joins, alternative="two-sided")
        sst_roughness.value = (
            f"<b>Join roughness, GO{block_size} vs GS{block_size}:</b> median {np.median(go_joins):.2f} vs "
            f"{np.median(gs_joins):.2f} ({len(go_joins)} vs {len(gs_joins)} join(s); frame-to-frame change in the span: median "
            f"{np.median(adjacent):.2f}). Mann-Whitney U = {test.statistic:.0f}, two-sided p = {test.pvalue:.3g}, "
            f"P(GS &gt; GO) = {test.statistic / (len(go_joins) * len(gs_joins)):.2f}.")
    else:
        sst_roughness.value = f"<b>GO{block_size} or GS{block_size} has no joins</b>, so there is nothing to compare."
    fig = make_subplots(
        rows=4, cols=4, specs=[[{"colspan": 4}, None, None, None], [{}] * 4, [{}] * 4, [{}] * 4],
        row_heights=[0.32, 0.28, 0.2, 0.2], vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=[
            f"{utterance}: {features.shape[0]} frames; span of {span} frames ({span * period:.2f} s) "
            f"in its middle, GO/GS gaps shaded (GO joins splice frames {(gap + 1) * period * 1000:.0f} ms apart)<br>"
            "frames each input takes, below: OC (blue), GC (red), GO/GS (green)",
            f"OC{block_size}: the span's contiguous middle",
            f"GC{block_size}: one cut of {cut} frames",
            f"GO{block_size}: {num_blocks} blocks, gaps of {gap} frames",
            f"GS{block_size}: GO's blocks shuffled",
            *[f"{name}{block_size}: source frame per position" for name in sst_strategies],
            *[f"{name}{block_size}: " + (f"{len(roughness[name])} join(s), median {np.median(roughness[name]):.2f}"
                                         if len(roughness[name]) else "no joins")
              for name in sst_strategies]])

    # One colour scale for the utterance and all inputs, so equal values get equal colours.
    fig.update_layout(coloraxis=dict(colorscale="RdBu_r", cmid=0, colorbar=dict(title_text="level", len=0.4, y=0.8)))
    fig.add_trace(go.Heatmap(
        z=features.T, x=np.arange(features.shape[0]) * period, y=np.arange(features.shape[1]), coloraxis="coloraxis",
        hovertemplate="time: %{x:.3f} s<br>bin: %{y}<br>level: %{z:.2f}<extra></extra>",
    ), row=1, col=1)
    for i in range(num_blocks - 1):
        gap_start = start + i * (block_size + gap) + block_size
        fig.add_vrect(x0=(gap_start - 0.5) * period, x1=(gap_start + gap - 0.5) * period, fillcolor="black",
                      opacity=0.35, line_width=0, row=1, col=1)
    for x in (start, start + span):
        fig.add_vline(x=(x - 0.5) * period, line=dict(color="black", width=1.5), row=1, col=1)
    # Below the spectrogram: a rug of the frames each input takes.
    colors = dict(OC="#1f77b4", GC="#d62728", GO="#2ca02c", GS="#2ca02c")
    for offset, name in enumerate(("OC", "GC", "GO")):
        taken = np.unique(sources[name])
        fig.add_trace(go.Scattergl(
            x=taken * period, y=np.full(len(taken), -4 - 6 * offset), mode="markers", showlegend=False,
            marker=dict(color=colors[name], size=5, symbol="line-ns-open"),
            hovertemplate=f"{name}: frame at %{{x:.3f}} s<extra></extra>",
        ), row=1, col=1)

    roughness_max = max(adjacent.max(), *(values.max() for values in roughness.values() if len(values)))
    roughness_bins = dict(start=0, end=roughness_max, size=roughness_max / 40)
    for col, name in enumerate(sst_strategies, start=1):
        segment, source = inputs[name], sources[name]
        fig.add_trace(go.Heatmap(
            z=segment.T, x=np.arange(segment.shape[0]), y=np.arange(segment.shape[1]), coloraxis="coloraxis",
            customdata=np.tile(source * period, (segment.shape[1], 1)),
            hovertemplate="position: %{x}<br>from: %{customdata:.3f} s<br>bin: %{y}<br>level: %{z:.2f}<extra></extra>",
        ), row=2, col=col)
        # Joins: where two frames meet that weren't neighbours in the utterance.
        for position in joins[name]:
            fig.add_vline(x=position - 0.5, line=dict(color="black", width=1, dash="dot"), row=2, col=col)
        fig.add_trace(go.Scattergl(
            x=np.arange(len(source)), y=source * period, mode="markers", marker=dict(color=colors[name], size=3),
            showlegend=False, hovertemplate="position: %{x}<br>from: %{y:.3f} s<extra></extra>",
        ), row=3, col=col)
        fig.update_xaxes(title_text="input position [frame]", row=3, col=col)
        # Join roughness, over the frame-to-frame change in the span (grey), on shared bins.
        fig.add_trace(go.Histogram(
            x=adjacent, xbins=roughness_bins, histnorm="probability", marker_color="grey", opacity=0.5,
            showlegend=False, hovertemplate="frame-to-frame: %{x}<br>share: %{y:.3f}<extra></extra>",
        ), row=4, col=col)
        if len(roughness[name]):
            fig.add_trace(go.Histogram(
                x=roughness[name], xbins=roughness_bins, histnorm="probability", marker_color=colors[name],
                opacity=0.7, showlegend=False, hovertemplate=f"{name} joins: %{{x}}<br>share: %{{y:.3f}}<extra></extra>",
            ), row=4, col=col)
        fig.update_xaxes(title_text="distance between frames", range=[0, roughness_max], row=4, col=col)
    fig.update_xaxes(title_text="time [s]", row=1, col=1)
    fig.update_yaxes(title_text="bin", row=1, col=1)
    fig.update_yaxes(title_text="bin", row=2, col=1)
    fig.update_yaxes(title_text="source time [s]", row=3, col=1)
    fig.update_yaxes(title_text="share", row=4, col=1)
    fig.update_layout(barmode="overlay", height=1250, margin=dict(l=60, r=80, t=60, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    sst_plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 1270px; border: 0"></iframe>'


sst_output = widgets.interactive_output(show_sst, dict(
    utterance=sst_utterance, front_end=sst_front_end, block_size=sst_block_size, gap_ratio=sst_gap_ratio,
    segment_duration=sst_segment_duration, seed=sst_seed,
))
display(widgets.VBox([
    widgets.HBox([sst_speaker, sst_utterance, sst_front_end]),
    widgets.HBox([sst_block_size, sst_gap_ratio, sst_segment_duration, sst_seed]),
    sst_plot,
    sst_roughness,
    sst_player,
    widgets.HBox(list(sst_input_players.values())),
    widgets.HBox(list(sst_resynth_players.values())),
    sst_output,
]))

## Block sizes in milliseconds

A block of b frames lasts b frame periods: 10 ms with WeSpeaker's fbank (ResNet34) and 20 ms with w2v-BERT 2.0's stacked fbank, so the same b covers twice the time with w2v-BERT. The signal a block covers is longer by the rest of its last frame's 25 ms window: (b − 1) · 10 + 25 ms with WeSpeaker, (2b − 1) · 10 + 25 ms with w2v-BERT, as each w2v-BERT frame stacks two 10 ms fbank frames.

| b | WeSpeaker: block | signal | w2v-BERT: block | signal |
|---|---|---|---|---|
| 2 | 20 ms | 35 ms | 40 ms | 55 ms |
| 4 | 40 ms | 55 ms | 80 ms | 95 ms |
| 8 | 80 ms | 95 ms | 160 ms | 175 ms |
| 16 | 160 ms | 175 ms | 320 ms | 335 ms |
| 32 | 320 ms | 335 ms | 640 ms | 655 ms |
| 64 | 640 ms | 655 ms | 1280 ms | 1295 ms |

For scale: a phone lasts about 50–100 ms, a syllable about 150–250 ms. With WeSpeaker, b = 8 is about a phone and b = 16–32 one to two syllables.

The gaps between GO's/GS's blocks at gap ratio 1.25 are round(1.25 · b) = 2, 5, 10, 20, 40, 80 frames: 20, 50, 100, 200, 400 and 800 ms with WeSpeaker, twice that with w2v-BERT. (Python's `round` rounds 2.5 to 2, so b = 2's gap is 2 frames, not 3.)

A 2 s segment is 196 frames with WeSpeaker and 98 with w2v-BERT, so b = 64 leaves ceil(196 / 64) = 4 blocks (3 joins) with WeSpeaker and 2 blocks (1 join) with w2v-BERT.

Over all VoxCeleb1-H utterances (cell above, WeSpeaker, gap ratio 1.25, seed 0), GO's joins are about as rough as GS's only from b = 32 on, where the gap is 40 frames (400 ms): frames that far apart are about as unrelated as two random frames. Below that, GS's joins are rougher (GS/GO median 1.98, 1.41, 1.14, 1.06 for b = 2, 4, 8, 16), so L(b) = EER(GS) − EER(GO) may contain join artefacts there, which push it up.


## GO vs GS join roughness over all VoxCeleb1-H utterances

The join roughness of the section above, over every utterance of the VoxCeleb1-H trial list rather than one, for every block size b. It uses the widgets' front-end, gap ratio, segment and seed at the time this cell runs, and draws each input exactly as the evaluation does (`trainer._eval_windows`: one window centred in the utterance, GS's block order from `default_rng(seed)`). Utterances shorter than b's gapped span have no input and are left out for that b, as their trials aren't scored. Featurizing the ~138k utterances takes a few minutes (all CPU cores).

- **Summary.** Per b, over all joins of all scored utterances (pooled): the mean, median and interquartile range of the distance between the two frames at a join, for GO and GS. Paired per utterance (GO and GS take the same blocks): the mean distance at GS's joins minus that at GO's, its median over utterances, the share of utterances where GS's joins are rougher, and a Wilcoxon signed-rank test of these differences. With this many utterances, even a tiny difference is significant, so read the effect sizes (GS/GO median, share GS > GO; 0.5 = equally rough) rather than the p-values. The line above the table gives the reference: the distance between neighbouring frames (every 10th pair of every utterance).
- **Distributions.** Per b, the densities of GO's and GS's join distances and of the frame-to-frame reference, on shared bins up to the 99.5th percentile.

If GS's joins are about as rough as GO's at a block size, the join artefacts cancel in L(b) = EER(GS) − EER(GO) there. Where GS's are clearly rougher, part of L(b) may be artefact; `doc/sst-roughness-strata.ipynb` checks whether that matters for the EER.


In [5]:
import os
import time

import pandas as pd
from scipy.stats import wilcoxon

from src.evaluation.temporal_probes import _join_distances
from src.training.trainer import resolved_utterances


def load_features(path, config):
    waveform, _ = sf.read(str(path), dtype="float32")
    return featurize_waveform(waveform, config)


# The settings of the widgets above, when this cell runs.
# (Named apart from the first section's widgets gap_ratio and seed.)
front_end, sst_ratio, sst_seed_value = sst_front_end.value, sst_gap_ratio.value, sst_seed.value
config = TransformationConfig(type=front_end, frame_length_s=0.025, frame_step_s=0.01)
segment_length = DataConfig(segment_duration=sst_segment_duration.value).segment_length(config)
names = [f"{prefix}{b}" for b in BLOCK_SIZES for prefix in ("GO", "GS")]
# Every input drawn with the same seed (as the final evaluation draws every strategy with default_rng(run_idx)),
# twice: once from the features and once from the frame indices, which locate the joins.
rngs = {name: (np.random.default_rng(sst_seed_value), np.random.default_rng(sst_seed_value)) for name in names}
joins = {name: [] for name in names}  # per utterance: the distances at its input's joins (empty: too short)
adjacent = []  # every 10th frame-to-frame distance of every utterance, the reference
entries = [(partial(load_features, vox1_root / "wav" / path, config), path) for path in vox1h_paths]
began = time.monotonic()
for i, (features, _) in enumerate(resolved_utterances(entries, num_workers=os.cpu_count())):
    adjacent.append(np.linalg.norm(np.diff(features, axis=0), axis=1)[::10])
    for name in names:
        distances = _join_distances(features, segment_length, segment_length, 1, name, rngs[name], sst_ratio)
        joins[name].append(distances[0] if distances else np.empty(0))
    if (i + 1) % 10000 == 0:
        print(f"{i + 1}/{len(entries)} utterances, {time.monotonic() - began:.0f} s")
adjacent = np.concatenate(adjacent)

# Summary per block size. Pooled: all joins of all utterances. Paired: per utterance, the mean distance at
# GS's joins minus at GO's, which take the same blocks; Wilcoxon signed-rank test of these differences.
rows = []
for b in BLOCK_SIZES:
    go_joins, gs_joins = joins[f"GO{b}"], joins[f"GS{b}"]
    scored = [i for i, (go_d, gs_d) in enumerate(zip(go_joins, gs_joins)) if len(go_d) and len(gs_d)]
    go_all = np.concatenate([go_joins[i] for i in scored])
    gs_all = np.concatenate([gs_joins[i] for i in scored])
    difference = np.array([gs_joins[i].mean() - go_joins[i].mean() for i in scored])
    rows.append({
        "b": b, "utterances": len(scored), "joins/utterance": len(go_all) // len(scored),
        "GO mean": go_all.mean(), "GO median": np.median(go_all),
        "GO IQR": np.subtract(*np.percentile(go_all, [75, 25])),
        "GS mean": gs_all.mean(), "GS median": np.median(gs_all),
        "GS IQR": np.subtract(*np.percentile(gs_all, [75, 25])),
        "GS/GO median": np.median(gs_all) / np.median(go_all),
        "paired diff median": np.median(difference), "utterances GS > GO": np.mean(difference > 0),
        "Wilcoxon p": wilcoxon(difference).pvalue,
    })
summary = pd.DataFrame(rows).set_index("b")
print(f"{front_end}, segment {segment_length} frames, gap ratio {sst_ratio}, seed {sst_seed_value}; "
      f"frame-to-frame distance: mean {adjacent.mean():.2f}, median {np.median(adjacent):.2f}, "
      f"IQR {np.subtract(*np.percentile(adjacent, [75, 25])):.2f}")
display(summary.style.format(precision=3).format({"Wilcoxon p": "{:.3g}"}))

# Distributions, on shared bins. Binned here (np.histogram) instead of by plotly, so the page doesn't carry the
# millions of distances.
upper = np.percentile(np.concatenate([adjacent, *(np.concatenate(joins[name]) for name in names)]), 99.5)
edges = np.linspace(0, upper, 81)
centers, width = (edges[:-1] + edges[1:]) / 2, edges[1] - edges[0]
fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.06, vertical_spacing=0.14,
                    subplot_titles=[f"b = {b}: {summary.loc[b, 'utterances']} utterances, "
                                    f"{summary.loc[b, 'joins/utterance']} joins each" for b in BLOCK_SIZES])
for k, b in enumerate(BLOCK_SIZES):
    row, col = k // 3 + 1, k % 3 + 1
    for label, values, color in [("frame-to-frame", adjacent, "grey"),
                                 ("GO joins", np.concatenate(joins[f"GO{b}"]), "#2ca02c"),
                                 ("GS joins", np.concatenate(joins[f"GS{b}"]), "#9467bd")]:
        density, _ = np.histogram(values, bins=edges, density=True)
        fig.add_trace(go.Bar(
            x=centers, y=density, width=width, marker_color=color, opacity=0.5, name=label, legendgroup=label,
            showlegend=k == 0, hovertemplate=f"{label}: %{{x:.2f}}<br>density: %{{y:.4f}}<extra></extra>",
        ), row=row, col=col)
    fig.update_xaxes(title_text="distance between frames", row=row, col=col)
fig.update_yaxes(title_text="density", col=1)
fig.update_layout(barmode="overlay", height=700, margin=dict(l=60, r=40, t=60, b=50),
                  title_text=f"Join roughness over VoxCeleb1-H, {front_end}, gap ratio {sst_ratio}")
page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
display(widgets.HTML(f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 720px; border: 0"></iframe>'))


10000/137924 utterances, 43 s


KeyboardInterrupt: 

## Extended utterances

A gapped span needs more audio than most VoxCeleb1-H utterances have: with a gap of about 450 ms, b = 8's span is ~13 s, b = 2's ~46 s (median utterance 6.5 s). Every utterance in a VoxCeleb1 video folder is of the same speaker, so a trial utterance can be extended by the other utterances of its video (median 93 s per trial utterance in all). The extended utterance is the trial utterance followed by the other utterances of its video, in file order; each join between two of them is a cut between two segments of the video that weren't adjacent.

Five trial utterances drawn at random, each played alone and extended (at their original levels, so level changes between the parts are audible). 32,778 of Vox1-H's trials (all target trials) pair two utterances of the same video; extended, both sides would contain the same audio, so these trials would have to be left out.


In [ ]:
def extended_utterance(path):
    # The trial utterance followed by the other utterances of its video (all the same speaker), in file order.
    # Returns the waveform and the relative paths of its parts, in order.
    own = vox1_root / "wav" / path
    parts = [own, *sorted(p for p in own.parent.glob("*.wav") if p != own)]
    waveforms = []
    for part in parts:
        waveform, sample_rate = sf.read(str(part), dtype="float32")
        assert sample_rate == 16000, f"{part} is {sample_rate} Hz"
        waveforms.append(waveform)
    return waveforms, [str(part.relative_to(vox1_root / "wav")) for part in parts]


# Five trial utterances drawn at random (fixed seed), each alone and extended.
examples = sorted(np.random.default_rng(0).choice(vox1h_paths, 5, replace=False))
rows = []
for path in examples:
    waveforms, parts = extended_utterance(path)
    starts = np.cumsum([0, *map(len, waveforms[:-1])]) / 16000
    extended = np.concatenate(waveforms)
    rows.append(widgets.HTML(
        f"<b>{path}</b>: {len(waveforms[0]) / 16000:.1f} s alone, {len(extended) / 16000:.1f} s extended "
        f"({len(parts)} utterance(s) of the video; the next ones start at "
        f"{', '.join(f'{s:.1f}' for s in starts[1:]) or '–'} s)"
        f"<br>alone{Audio(waveforms[0], rate=16000, normalize=False)._repr_html_()}"
        f"extended{Audio(extended, rate=16000, normalize=False)._repr_html_()}"))
display(widgets.VBox(rows))
